# A reproducible scientific workflow

This offline notebook uses a generated dataset and bounded settings. It demonstrates data profiling, a small AutoML search, repeated nested-CV evaluation, and model-specific feature interpretation. No download, API service, or external dataset is required.

Run from the repository root with `jupyter lab examples/notebooks/scientific_workflow.ipynb`. Results are examples, not scientific conclusions or model-quality guarantees.

In [ ]:
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.metrics import get_scorer
from sklearn.model_selection import train_test_split

from SKSurrogate import AML, dataset_profile, mltrack

SEED = 7
X, y = make_classification(
    n_samples=240,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    random_state=SEED,
)
features = [f"feature_{index}" for index in range(X.shape[1])]
frame = pd.DataFrame(X, columns=features).assign(target=y)
train, validation = train_test_split(
    frame, test_size=0.25, random_state=SEED, stratify=frame["target"]
)
train = train.reset_index(drop=True)
validation = validation.reset_index(drop=True)
print(f"Training rows: {len(train)}; validation rows: {len(validation)}")

## Inspect data before modeling

The profile reports distributions, missingness, and exploratory feature/target associations. Associations are diagnostic signals, not proof of causation, leakage, or redundant scientific mechanisms.

In [ ]:
profile = dataset_profile(
    train,
    target="target",
    max_rows=500,
    max_features=10,
    max_categories=10,
    random_state=SEED,
)
display(pd.DataFrame(profile["columns"])[["column", "kind", "missing_rate", "unique_count"]])
display(pd.DataFrame(profile["target_associations"]))

## Search with a bounded preset

The classification preset uses scikit-learn estimators and a small candidate budget. The validation partition stays out of the search and is used only for the final score.

In [ ]:
search_artifacts = tempfile.TemporaryDirectory(prefix="sksurrogate-search-")
search = AML(
    preset="classification",
    max_evals=5,
    time_limit=30,
    cv=3,
    random_state=SEED,
    n_jobs=1,
    check_point=str(Path(search_artifacts.name)) + "/",
    verbose=0,
)
search.fit(train[features].to_numpy(), train["target"].to_numpy())
best_model = search.best_estimator_
validation_score = best_model.score(validation[features].to_numpy(), validation["target"].to_numpy())
print(f"Selected model: {best_model.__class__.__name__}")
print(f"Held-out validation score: {validation_score:.3f}")

## Evaluate and record the run

Repeated nested CV summarizes variability across repeat-level outer-fold means. With only two repeats, its normal-approximation interval is illustrative and should not be treated as precise. The validation partition remains separate from training-based model selection.

In [ ]:
workdir = tempfile.TemporaryDirectory(prefix="sksurrogate-notebook-")
tracker = mltrack(
    "notebook-scientific-workflow",
    db_name=str(Path(workdir.name) / "mltrace.db"),
)
tracker.RegisterData(train, "target", partition="train")
tracker.RegisterData(validation, "target", partition="validation")
evaluation = tracker.nested_cv_evaluation(
    best_model,
    inner_cv=2,
    outer_cv=2,
    train_partition="train",
    validation_partition="validation",
    scoring=get_scorer("accuracy"),
    repeats=2,
    confidence_level=0.95,
)
print("Repeated outer scores:", evaluation["repeated_outer_scores"])
print("Confidence interval:", evaluation["confidence_interval"])
print("Final validation score:", evaluation.get("final_validation_score"))

## Interpret the fitted model

Permutation importance is computed on the explicitly registered validation partition, with a fixed seed and bounded repeats. It describes this model on this reference sample; it is not a causal effect. Close the tracker and temporary database when finished.

In [ ]:
logged_model = tracker.LogModel(best_model, "bounded-preset-classifier")
importance = tracker.permutation_importance(
    logged_model,
    reference_partition="validation",
    n_repeats=3,
    max_samples=60,
    random_state=SEED,
)
display(importance["importances"][["feature", "importance_mean", "importance_std"]])
print("Reference:", importance["reference"])
tracker.close()
workdir.cleanup()
search_artifacts.cleanup()